# XGBoost Multi-Quantile Benchmark

Benchmarks `likelihood="quantile"` (one model per quantile) vs `likelihood="multiquantile"` (single model) for `XGBModel`.

The speedup comes from Darts' `_create_lagged_data` (tabularization) running once instead of N times.

Related: issue #3047, branch `feat/xgb-multiquantile`.

In [2]:
import time

import numpy as np

from darts.datasets import AirPassengersDataset, ETTh1Dataset
from darts.models import XGBModel

N_REPEATS = 5


def bench(series, likelihood, n_repeats=N_REPEATS, **kwargs):
    times = []
    for _ in range(n_repeats):
        model = XGBModel(likelihood=likelihood, **kwargs)
        t0 = time.perf_counter()
        model.fit(series)
        times.append(time.perf_counter() - t0)
    return np.mean(times), np.std(times)

## 1. Baseline — AirPassengers, 3 quantiles

In [3]:
series = AirPassengersDataset().load()
kwargs = dict(
    lags=12,
    output_chunk_length=6,
    quantiles=[0.1, 0.5, 0.9],
    n_estimators=100,
    random_state=42,
)

for likelihood in ["quantile", "multiquantile"]:
    mean, std = bench(series, likelihood, **kwargs)
    print(f"{likelihood:>15}: {mean:.2f}s ± {std:.3f}s")

       quantile: 0.78s ± 0.020s
  multiquantile: 0.76s ± 0.009s


## 2. Scale — ETTh1 (~17k pts), 11 quantiles

In [8]:
long_series = ETTh1Dataset().load()["OT"]
xd = [float(x) for x in np.round(np.linspace(0.05, 0.95, 11), 2)]
kwargs = dict(
    lags=24, output_chunk_length=12, quantiles=xd, n_estimators=50, random_state=42
)

for likelihood in ["quantile", "multiquantile"]:
    mean, std = bench(long_series, likelihood, n_repeats=3, **kwargs)
    print(f"{likelihood:>15}: {mean:.2f}s ± {std:.3f}s")

       quantile: 15.95s ± 0.159s
  multiquantile: 14.93s ± 0.154s


## 3. Tabularization stress — 19 quantiles, 168 lags, minimal fitting

In [9]:
# somehow crashes with np.float64 dtype for multiquantile
xd2 = [float(x) for x in np.round(np.linspace(0.05, 0.95, 19), 2)]
kwargs = dict(
    lags=168,
    output_chunk_length=24,
    quantiles=xd2,
    n_estimators=1,
    max_depth=1,
    random_state=42,
)

for likelihood in ["quantile", "multiquantile"]:
    mean, std = bench(long_series, likelihood, n_repeats=3, **kwargs)
    print(f"{likelihood:>15}: {mean:.2f}s ± {std:.3f}s")

       quantile: 21.36s ± 0.067s
  multiquantile: 2.29s ± 0.021s
